In [2]:
import os
import json
import numpy as np
from scipy.sparse import csr_matrix, save_npz
from collections import Counter

stats_dir = '../Datasets/decade_stats'
cleaned_dir = '../Datasets/decade_cleaned'
output_dir = '../Datasets/decade_matrices'
os.makedirs(output_dir, exist_ok=True)

# PPMI function
def build_hamilton_ppmi(tokens, vocab, window_size=4, neg=5):
    w2i = {word: i for i, word in enumerate(vocab)}
    vocab_size = len(vocab)
    
    # Create raw co-occurrence counts
    counts = Counter()
    for i, token in enumerate(tokens):
        if token not in w2i: continue
        t_idx = w2i[token]
        for j in range(max(0, i - window_size), min(len(tokens), i + window_size + 1)):
            if i == j or tokens[j] not in w2i: continue
            counts[(t_idx, w2i[tokens[j]])] += 1

    # Marginal probabilities with CDS (0.75 power Appendix A)
    row_sums = np.zeros(vocab_size)
    col_sums = np.zeros(vocab_size)
    for (r, c), val in counts.items():
        row_sums[r] += val
        col_sums[c] += val
    
    col_sums = np.power(col_sums, 0.75) 
    row_probs = row_sums / row_sums.sum()
    col_probs = col_sums / col_sums.sum()
    total_sum = sum(counts.values())

    # Build cparse matrix
    rows, cols, data = [], [], []
    for (r, c), val in counts.items():
        joint_prob = val / total_sum
        denom = row_probs[r] * col_probs[c]
        if denom > 0:
            # PMI - log(5) (Equation 1)
            pmi = np.log(joint_prob / denom) - np.log(neg)
            ppmi = max(pmi, 0) # The 'Positive' part in PPMI
            if ppmi > 0:
                rows.append(r); cols.append(c); data.append(ppmi)
    
    return csr_matrix((data, (rows, cols)), shape=(vocab_size, vocab_size))

# Loop through every decade
for vocab_file in sorted(os.listdir(stats_dir)):
    if vocab_file.endswith("_vocab.txt"):
        decade = vocab_file.replace("_vocab.txt", "")
        print(f"Processing {decade}...")

        with open(os.path.join(stats_dir, vocab_file), 'r', encoding='utf-8') as f:
            vocab = [line.strip() for line in f]
        with open(os.path.join(cleaned_dir, f"{decade}.txt"), 'r', encoding='utf-8') as f:
            tokens = f.read().split()

        matrix = build_hamilton_ppmi(tokens, vocab)
        save_npz(os.path.join(output_dir, f"{decade}_ppmi.npz"), matrix)

print("All decade PPMI matrices created!")

Processing 1810s...
Processing 1820s...
Processing 1830s...
Processing 1840s...
Processing 1850s...
Processing 1860s...
Processing 1870s...
Processing 1880s...
Processing 1890s...
Processing 1900s...
Processing 1910s...
Processing 1920s...
Processing 1930s...
Processing 1940s...
Processing 1950s...
Processing 1960s...
Processing 1970s...
Processing 1980s...
Processing 1990s...
All decade PPMI matrices created!
